In [84]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import numpy as np
import math

device  =  'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {device}')

#CONSTANTS:
batch_size = 64
block_size = 64
embed_count = 128
dropout_percent = 0.2

Device: cuda


In [85]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-10-22 05:28:36--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.9’

input.txt.9         100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-10-22 05:28:36 (37.3 MB/s) - ‘input.txt.9’ saved [1115394/1115394]



In [86]:
text = ''
vocab = []
with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()
  vocab = sorted(list(set(text)))

vocab
vocab_length = len(vocab)

print(f'Voculabulary: {vocab}')
print(f'letter count: {vocab_length}')

Voculabulary: ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
letter count: 65


In [87]:
char_to_token_dict = { c:t for t , c in enumerate(vocab)}
token_to_char_dict = { t:c for t, c in enumerate(vocab)}

print('Vocabs: \n')
print(char_to_token_dict)
print(token_to_char_dict)

encode = lambda t : [char_to_token_dict[c] for c in t]
print(f'\nEncoder test: {encode('hello guys')}')

decode = lambda s : ''.join([token_to_char_dict[t] for t in s])
print(f'\nDecoder test: {decode(encode('hello guys'))}')

Vocabs: 

{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}
{0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40: 'b', 41: 'c', 42: 'd', 43: 'e', 44: 'f', 45: 'g', 46: 'h'

In [88]:
data_tensor = torch.tensor(encode(text), dtype=torch.long)

data_len = len(data_tensor)

print(f'Data tensor of len: {data_len}')

split_count = int(0.8*data_len)
train_data = data_tensor[:split_count]
test_data = data_tensor[split_count:]

print(f'Train data size: {len(train_data)}')
print(f'Test data size: {len(test_data)}')

def get_batch(b_type):
  batch_pool = train_data if b_type == 'train' else test_data if b_type == 'test' else None
  if(batch_pool == None):
    print('INVALID BATCH TYPE, RETURNING -1')
    return -1, -1
  random_index = torch.randint(data_len - block_size, (batch_size,))
  x = torch.stack([data_tensor[i:i+block_size] for i in random_index]).to(device)
  y = torch.stack([data_tensor[i+1:i+block_size+1] for i in random_index]).to(device)
  return x,y

example_x, example_y  = get_batch('train')
print('\nTesting get batch')
print(f'Example x: {example_x}')
print(f'Example y: {example_y}')

Data tensor of len: 1115394
Train data size: 892315
Test data size: 223079

Testing get batch
Example x: tensor([[53, 59, 56,  ..., 59, 56,  1],
        [ 1, 58, 46,  ..., 61, 39, 56],
        [43, 56, 43,  ..., 39, 42, 43],
        ...,
        [53, 53, 49,  ..., 33, 31, 10],
        [53, 50, 50,  ..., 51,  1, 42],
        [ 1, 40, 39,  ..., 56, 43, 44]], device='cuda:0')
Example y: tensor([[59, 56,  1,  ..., 56,  1, 46],
        [58, 46, 39,  ..., 39, 56, 42],
        [56, 43,  1,  ..., 42, 43,  1],
        ...,
        [53, 49,  6,  ..., 31, 10,  0],
        [50, 50, 53,  ...,  1, 42, 43],
        [40, 39, 63,  ..., 43, 44, 53]], device='cuda:0')


In [103]:
class CausalAttentionLayer(nn.Module):
  def __init__(self, embed_count, head_count):
    super().__init__()
    assert embed_count % head_count == 0
    self.embed_count = embed_count
    self.head_count = head_count
    self.embed_per_head = embed_count // head_count # since this is a sinpler model, for each head of attention we will split the data

    self.qkv_op = nn.Linear(embed_count, 3*embed_count, bias= False) # instead of tediously mutliplying each embed with Q, K, V,we
    self.mix = nn.Linear(embed_count, embed_count, bias = False)
    self.dropout = nn.Dropout(dropout_percent)

  def forward(self, x):
    B, T, C = x.shape

    QKV = self.qkv_op(x)
    Q, K, V = QKV.split(C, dim=2)

    Q = Q.view(B, T, self.head_count, self.embed_per_head).transpose(1,2)
    K = K.view(B, T, self.head_count, self.embed_per_head).transpose(1,2)
    V = V.view(B, T, self.head_count, self.embed_per_head).transpose(1,2)

    # try to make this from scratch later
    attention_result = F.scaled_dot_product_attention(Q, K, V, is_causal = True, dropout_p=dropout_percent if self.training else 0)

    attention_result = attention_result.transpose(1, 2).contiguous().view(B, T, C)
    attention_result = self.mix(attention_result)
    attention_result = self.dropout(attention_result)

    return attention_result

class FeedForward(nn.Module): # the intuition for the feed forward layers is "fact storage." Or mixing embed info with fact info
  def __init__(self, embed_count):
    super().__init__()
    self.embed_count = embed_count
    self.network = nn.Sequential(
        nn.Linear(embed_count, embed_count*4), # this should be projected over each embed vector
        nn.GELU(), # industry standard
        nn.Linear(embed_count*4, embed_count),
        nn.Dropout(dropout_percent)
    )

  def forward(self, x):
    return self.network(x)

class Block(nn.Module):
  def __init__(self, embed_count, head_count):
    super().__init__()
    self.embed_count = embed_count
    self.head_count = head_count
    self.normalize_1 = nn.LayerNorm(embed_count)
    self.attention_layer = CausalAttentionLayer(embed_count, head_count)
    self.normalize_2 = nn.LayerNorm(embed_count)
    self.feed_forward_layer = FeedForward(embed_count)

  def forward(self, x):
    result = self.normalize_1(x)
    result = result + self.attention_layer(result)
    result = result + self.feed_forward_layer(self.normalize_2(result))
    return result

class MiniGPT(nn.Module):
  def __init__(self, vocab_count, embed_count, head_count, layer_count):
    super().__init__()
    self.vocab_count = vocab_count
    self.embed_count = embed_count
    self.head_count = head_count
    self.layer_count = layer_count

    self.embedder = nn.Embedding(vocab_count, embed_count)
    self.pos_embedder = nn.Embedding(block_size, embed_count)
    self.block_sequence = nn.Sequential(*[Block(embed_count, head_count) for c in range(layer_count)])
    self.final_normalize = nn.LayerNorm(embed_count)
    self.unembed = nn.Linear(embed_count, vocab_count, bias = False)
    self.embedder.weight = self.unembed.weight

    self.apply(self.initialize_weights)

    for name, p in self.named_parameters():
          if name.endswith('mix.weight'):
              torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * layer_count))

  def initialize_weights(self, module):
      if isinstance(module, nn.Linear):
          torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
          if module.bias is not None:
              torch.nn.init.zeros_(module.bias)
      elif isinstance(module, nn.Embedding):
          torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

  def forward(self, x, target=None):
    B, T = x.shape
    x = self.embedder(x)
    position_embeds = self.pos_embedder(torch.arange(T, device=device))
    x = x + position_embeds

    x = self.block_sequence(x)
    x = self.final_normalize(x)
    logits = self.unembed(x)

    if target is None:
      return logits, None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      target = target.view(B*T)
      loss = F.cross_entropy(logits, target)
      return logits, loss

  def generate(self, full_context, new_tokens, top_k = 30, temperature = 0.8):
    for _ in range(new_tokens):
      #shape = B*T, B = 1 (Unsqueezed data)
      block_context = full_context[:, -block_size:]
      logits, _ = self.forward(block_context) # shape: B T C
      logits = logits[:, -1, :]/temperature # take the prediction for the last token for each batch (or in this case 1 batch), So the shape is B, C, with B = 1 usually (note:, we cans till have mutliple batches for generation too, no hard and fast rule, this is for my own visualization)

      # now we wanna take top k predictions, set the rest to - infitnity. They will then be made irrelevant when applying softmax

      if(top_k is not None):
        least_vals, _ = torch.topk(logits, min(top_k, logits.size(-1)) ) # since logits is B C, least vals is B top_k
        logits[logits < least_vals[:, [-1]]] = float('-inf')# loast_vals[: [-1]] has shape B , 1

      preds = torch.softmax(logits, dim= -1) # Shape B, C, along C we now have probabilites. Vast majority are now 0
      next_token = torch.multinomial(preds, 1) # B, 1, for each batch it has the token id for the next best token. Note that since th last dimension is of form token id, the shape is now B, T, with T = 1
      full_context = torch.cat([full_context, next_token], dim=1) # B, T+1, Type B T
      generated_char = decode(next_token[0].tolist())
      print(generated_char, end='')
    return full_context


model = MiniGPT(vocab_length, 1280, 4, 6).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Model parameters: 118,201,600


In [104]:


def cosine_lr_controller(lr, current_epoch, max_epochs):
  min_lr = lr*0.1
  warm_up_epochs = 50
  if(current_epoch< warm_up_epochs):
    return lr * (current_epoch+1)/warm_up_epochs # smooth rising of lr to max lr in a linear fashion. We add one so we never hve a 0 learning rate
  else:
    pi_coeff = (current_epoch-warm_up_epochs)/(max_epochs - warm_up_epochs) # coeff moves from 0 to 1 linearly as we move form wamr up epochs to max epochs
    # we will use this value to smoothly move down lr to min lr
    cos_coeff = 0.5 *(1+ math.cos(math.pi*pi_coeff)) # trasnsions from 1 to 0 around a smooth and graceful curve
    return (lr-min_lr)*cos_coeff + min_lr # smooth transion from lr to min_lr



In [105]:
@torch.no_grad()
def evaluate(model, iterations):
  outputs = []
  model.eval()
  for j, data_set in enumerate(['train', 'test']):
    losses = torch.zeros(iterations)
    for i in range(iterations):
      xb, yb = get_batch(data_set)
      logits, loss = model.forward(xb, yb)
      losses[i] = loss
    mean_loss = losses.mean().item()
    outputs.append(mean_loss) # Fixed: use append instead of indexing an empty list
  model.train()
  return outputs

learning_rate = 4e-4
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate, betas = (0.9, 0.95), weight_decay = 0.1)
epochs = 1500

for epoch in range(epochs):
  effective_lr = cosine_lr_controller(learning_rate, epoch, epochs)
  for param_group in optimizer.param_groups:
    param_group['lr'] = effective_lr # research param groups later
  xb, yb = get_batch('train')
  logits, loss = model(xb, yb)

  if epoch % 25 == 0:
    losses = evaluate(model, 50)
    print(f'FOR EPOCH {epoch}, TRAIN LOSS: {losses[0]}, TEST LOSS: {losses[1]}')

  optimizer.zero_grad()
  loss.backward()
  torch.nn.utils.clip_grad_norm_(model.parameters(), 1) # trunicates gradients to prevent massive gradient epxlosisosn
  optimizer.step()

checkpoint = {
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss.item(),
}

torch.save(checkpoint, 'checkpoint.pt')

FOR EPOCH 0, TRAIN LOSS: 6.6394944190979, TEST LOSS: 6.6378397941589355
FOR EPOCH 25, TRAIN LOSS: 2.656785249710083, TEST LOSS: 2.656019687652588
FOR EPOCH 50, TRAIN LOSS: 2.4163639545440674, TEST LOSS: 2.4124350547790527
FOR EPOCH 75, TRAIN LOSS: 2.2429773807525635, TEST LOSS: 2.245986223220825
FOR EPOCH 100, TRAIN LOSS: 2.1240105628967285, TEST LOSS: 2.116319179534912
FOR EPOCH 125, TRAIN LOSS: 2.008979558944702, TEST LOSS: 2.015889883041382
FOR EPOCH 150, TRAIN LOSS: 1.9402683973312378, TEST LOSS: 1.9359207153320312
FOR EPOCH 175, TRAIN LOSS: 1.874863862991333, TEST LOSS: 1.8774960041046143
FOR EPOCH 200, TRAIN LOSS: 1.8160263299942017, TEST LOSS: 1.8203332424163818
FOR EPOCH 225, TRAIN LOSS: 1.7866522073745728, TEST LOSS: 1.78355073928833
FOR EPOCH 250, TRAIN LOSS: 1.7578223943710327, TEST LOSS: 1.7576316595077515
FOR EPOCH 275, TRAIN LOSS: 1.7201666831970215, TEST LOSS: 1.7149981260299683
FOR EPOCH 300, TRAIN LOSS: 1.6900941133499146, TEST LOSS: 1.68668794631958
FOR EPOCH 325, TRA

In [101]:
prompt = 'MARQUES: \nWhere thoust stay?'
context = torch.tensor(encode(prompt), dtype=torch.long, device=device) # shape is T


generated_chars = decode(model.generate(context.unsqueeze(0), new_tokens=1000, temperature=0.8, top_k=40)[0].tolist())




BUCKINGHAM:
And me, you lords not bloody
That I she child: i' a night with the breather deady,
In she my laughter with a tears and my heart:
Musicitizener, thou sir; all the man.

AUTOLYCUS:
You are your lay, when dones:
WhI have my father marriage cut loves me to amy
Before, by a usin peouty, thou ple's sout,
Live so suffence and he!

MENENIUS:
All pone; I'll buight the again, and friely earth.

CLARENCE:
That and you use to seement ready: as death sin
From there to me. Here is crift, you may to the godler.

NORTHUMBERLAND:
Let entent me to the the lands arm, there's doubles
In a king to him I am ragely
The aary in my Many mineral agage anged.

ROMEO:
Thee, for my ne god bud a stard with a sull,
The words decented with the nexch'd a counself
Two many pass'd in refear us the so?

FRIAR LAURENCE:
If I know it is our wonder arms under him her,
And deated themmataint o' to the should so than, what we do I
The shephedge the pince of the takes, abouse
With my faced uncless and of his bear